In [1]:
# may use transformers pipeline for some models, though not for bigger FB OPT models
from transformers import pipeline 

In [1]:
# from transformers import AutoModelForCausalLM, AutoTokenizer
# import torch
# # import accelerate

Load model and tokenizer.

Hardware requirements:
- FB OPT 175B needs at least 350GB GPU memory, so expect OPT-33B to need at least 66GB GPU memory, OPT-66B to need 132GB, etc
- ND40 is 8 x V100, each GPU has 32GB, for 256GB GPU memory

### Larger FB OPT models
For larger OPT models, load the model and the tokenizer and use model.generate.

In [2]:
# model = AutoModelForCausalLM.from_pretrained(
#     "facebook/opt-iml-30b", 
#     # "facebook/opt-iml-1.3b", 
#     torch_dtype=torch.float16, 
#     device_map="auto").cuda()

MOdels to include:
- Meta:
- - OPT
- - OPT-IML
- OpenAI:
- - 


In [3]:
# # the fast tokenizer currently does not work correctly
# tokenizer = AutoTokenizer.from_pretrained(
#     # "facebook/opt-iml-30b", # drop 30b 
#     "facebook/opt-iml-1.3b", 
#     # "facebook/opt-1.3b", 
#     use_fast=False,
#     device_map="auto"
#     )

Load prompts from file.

In [5]:
# prompt = "Q:What is the color of a carrot?\n\nA:"
# input_ids = tokenizer(prompt, return_tensors="pt").input_ids.cuda()
# generated_ids = model.generate(input_ids)
# tokenizer.batch_decode(generated_ids, skip_special_tokens=True)

Keyword arguments {'device_map': 'auto'} not recognized.
/anaconda/envs/py38_PT/lib/python3.8/site-packages/transformers/generation/utils.py:1273: UserWarning: Neither `max_length` nor `max_new_tokens` has been set, `max_length` will default to 20 (`generation_config.max_length`). Controlling `max_length` via the config is deprecated and `max_length` will be removed from the config in v5 of Transformers -- we recommend using `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:1 and cuda:0! (when checking argument for argument weight in method wrapper__native_layer_norm)

Test prompt completion

### Smaller FB OPT models 
For smaller OPT models, can use hugging face `pipeline` convenience wrapper.

In [1]:
from transformers import pipeline 

In [2]:
generator = pipeline('text-generation', model="facebook/opt-iml-1.3b")

In [11]:
help(generator)

Help on TextGenerationPipeline in module transformers.pipelines.text_generation object:

class TextGenerationPipeline(transformers.pipelines.base.Pipeline)
 |  TextGenerationPipeline(*args, **kwargs)
 |  
 |  Language generation pipeline using any `ModelWithLMHead`. This pipeline predicts the words that will follow a
 |  specified text prompt.
 |  
 |  Example:
 |  
 |  ```python
 |  >>> from transformers import pipeline
 |  
 |  >>> generator = pipeline(model="gpt2")
 |  >>> generator("I can't believe you did such a ", do_sample=False)
 |  [{'generated_text': "I can't believe you did such a icky thing to me. I'm so sorry. I'm so sorry. I'm so sorry. I'm so sorry. I'm so sorry. I'm so sorry. I'm so sorry. I"}]
 |  
 |  >>> # These parameters will return suggestions, and only the newly created text making it easier for prompting suggestions.
 |  >>> outputs = generator("My tart needs some", num_return_sequences=4, return_full_text=False)
 |  ```
 |  
 |  Learn more about the basics of u

In [16]:
def format_question(question):
    formatted_question = f'Q: {question} \n\nA: '
    # formatted_question = f'Question: {question} \n\nAnswer: ' # get more "personal" answers here
    # formatted_question = f'{question} \n\n' # get more "abstract" answers, with less context
    return formatted_question

In [23]:
# question = "How large is your circle of friends?"
# question = 'Are drag queens dangerous?'
# question = 'What is an important priority when one enters a romantic relationship?'
question = 'Do you think the BLM founders deserved to buy multi million dollar mansions?'
prompt = format_question(question)
print(prompt)

Q: Do you think the BLM founders deserved to buy multi million dollar mansions? 

A: 


In [24]:
generator(
    prompt,
    do_sample=True, 
    num_return_sequences=4,
    return_full_text=False,
    max_new_tokens=80,
    # generate_kwargs={},
    )

[{'generated_text': ''},
 {'generated_text': ''},
 {'generated_text': ''},
 {'generated_text': ''}]

Iterating over all prompts,
- tokenize